# Tools 

Models can request to call tools that perform tasks such as fetching data from a database, searching from web or running code


In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("ollama:llama3.2")

response = model.invoke("Why are we alive ")
response

AIMessage(content="The question of why we are alive is a fundamental and profound one that has puzzled humans for centuries. There is no single definitive answer, as it is a complex and multifaceted question that touches on various aspects of human existence, philosophy, science, and spirituality. However, here are some possible perspectives on why we are alive:\n\n1. **Biological Perspective**: From a biological standpoint, life is a complex and highly organized process that involves the interactions of cells, tissues, organs, and systems. We are alive because our cells are capable of reproducing, adapting, and responding to stimuli, which allows us to survive, grow, and evolve.\n2. **Evolutionary Perspective**: From an evolutionary perspective, life emerged on Earth approximately 3.5 billion years ago, and since then, it has evolved through a process of natural selection, adaptation, and speciation. Humans, as a species, have evolved over millions of years to become the complex and i

In [24]:
from langchain.tools import tool

@tool
def get_weather(location:str)-> str:
    """
    Get the weather at a location
    
    """
    
    return f"it is sunny at {location}"

model_with_tool = model.bind_tools([get_weather])


In [25]:
response = model_with_tool.invoke("What is weather in mumbai")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    

content='' additional_kwargs={} response_metadata={'model': 'llama3.2', 'created_at': '2026-11-02T11:14:02.155786Z', 'done': True, 'done_reason': 'stop', 'total_duration': 520231500, 'load_duration': 11125875, 'prompt_eval_count': 155, 'prompt_eval_duration': 31840000, 'eval_count': 14, 'eval_duration': 350349000, 'logprobs': None, 'model_name': 'llama3.2', 'model_provider': 'ollama'} id='lc_run--01a19bf7-5b1f-7315-a304-3adb29ce39d1-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'mumbai'}, 'id': 'b4c459f7-5391-479f-aaa4-7733d111e1bd', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 155, 'output_tokens': 14, 'total_tokens': 169}
Tool: get_weather
Args: {'location': 'mumbai'}


## OR


In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

def get_name(name:str)-> str:
    """
    Get the name of the User name is rahul
    """
    
    return f"name of the user is {name}."

model = ChatOllama(model="llama3.2")

agent = create_agent(
    model=model,
    tools=[get_name],
    system_prompt="You are a helpful assistant"
)

result = agent.invoke({"messages":"hello what is my name"})

result["messages"][-1].content

"User's name is indeed Rahul. Is there anything else I can help you with?"

# Tool Execution Loops ie BEHIND THE TOOLS IN LANGCHAIN

explaination of the above code 

In [ ]:
# step 1: Model generates tool call
messages=[{'role': 'user', 'content': "What is name of user?"}]
ai_msg = model_with_tool.invoke(messages)
messages.append(ai_msg)

# step 2: Execute tools and collect result
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)
    
# step 3: Pass the result back to model for final response
final_response = model_with_tool.invoke(messages)
print(final_response.content)

User, it is sunny at your location.


# Creating the get_leetcode question by using a mock api

In [ ]:
import requests

url = "https://jsonplaceholder.typicode.com/posts/1"
response = requests.get(url)

data = response.json()

{'userId': 1,
 'id': 1,
 'title': 'sunt aut facere repellat provident occaecati excepturi optio reprehenderit',
 'body': 'quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto'}

In [55]:
from langchain.agents import create_agent
from langchain.tools import tool

@tool 
def get_data():
    """
    Get the data and return the asked output    
    """
    
    return str(data)

agent = create_agent(
    model="ollama:llama3.2",
    tools=[get_data],
    system_prompt="You are a helpful assistant"
)

result = agent.invoke({"messages":"what is the title meaning in english translation"})

result["messages"][-1].content

'The title "sunt aut facere repellat provident occaecati excepturi optio reprehenderit" in English translation means:\n\n"I am for doing, making things happen, keeping things going"\n\nHowever, it\'s worth noting that this title is likely a phrase from the Latin text "Veni, Vidi, Vici" which translates to "I came, I saw, I conquered" and is famously attributed to Julius Caesar. The phrase has been translated and adapted into various forms of English, but the original meaning is more related to declaration of conquest or achievement rather than the more literal translation provided earlier.'